<a href="https://colab.research.google.com/github/bandapraneeth/IEEE/blob/main/IEEE_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, roc_curve, f1_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
import shap
import numpy as np

# ===================== Load Dataset =====================
df = pd.read_csv("/content/Students Social Media Addiction.csv")

# ===================== EDA =====================
sns.pairplot(df[["Avg_Daily_Usage_Hours", "Sleep_Hours_Per_Night", "Addicted_Score", "Affects_Academic_Performance"]],
             hue="Affects_Academic_Performance", palette="coolwarm")
plt.suptitle("Pairwise Relationships", y=1.02)
plt.show()

plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x="Sleep_Hours_Per_Night", hue="Affects_Academic_Performance", fill=True)
plt.title("Sleep Patterns by Academic Performance Impact")
plt.show()

# ===================== Machine Learning =====================
# NOVELTY: Baseline Accuracy (benchmark against majority class)
baseline = df["Affects_Academic_Performance"].value_counts(normalize=True).max()
print("Baseline Accuracy (majority class):", round(baseline * 100, 2), "%")

y = df["Affects_Academic_Performance"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Student_ID", "Affects_Academic_Performance"])

# Encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
encoder = LabelEncoder()
for col in categorical_cols:
    X[col] = encoder.fit_transform(X[col])

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Keep DataFrame version for SHAP & explainability
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (Linear)": SVC(kernel="linear", probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train & evaluate models
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print(f"\n{name} Results:")
    print("Accuracy:", round(acc * 100, 2), "%")
    print("Balanced Accuracy:", round(bal_acc * 100, 2), "%")   # NOVELTY: Balanced accuracy for imbalanced data
    print("F1-score:", round(f1 * 100, 2), "%")                 # NOVELTY: F1-score for imbalanced data
    print(classification_report(y_test, y_pred))

    # NOVELTY: Cross-validation for robust evaluation
    scores = cross_val_score(model, X_scaled, y, cv=5)
    print("Cross-validation Accuracy:", round(scores.mean() * 100, 2), "%")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # NOVELTY: ROC Curve & AUC (goes beyond accuracy)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")
    plt.plot([0,1],[0,1],'--',color='gray')
    plt.title(f"ROC Curve - {name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

    # NOVELTY: Learning Curve (detects overfitting/underfitting)
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_scaled, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1,1,5)
    )
    plt.figure(figsize=(6,4))
    plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label="Train")
    plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label="Validation")
    plt.title(f"Learning Curve - {name}")
    plt.xlabel("Training Size")
    plt.ylabel("Score")
    plt.legend()
    plt.show()

# ===================== Feature Importance =====================
# NOVELTY: Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8,5))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.show()

# NOVELTY: Permutation Importance (model-agnostic, robust)
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm.importances_mean.argsort()
plt.figure(figsize=(8,5))
plt.barh(X.columns[sorted_idx], perm.importances_mean[sorted_idx])
plt.title("Permutation Importance")
plt.show()

# ===================== PCA Visualization =====================
# NOVELTY: PCA for dimensionality reduction & visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette="coolwarm")
plt.title("PCA Projection of Students Data")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# ===================== SHAP Explainability =====================
# NOVELTY: SHAP values to interpret model predictions
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_df)

# NOVELTY: SHAP summary plot shows global feature importance with impact
shap.summary_plot(shap_values, X_test_df, feature_names=X.columns)

# ===================== Clustering Analysis =====================
# NOVELTY: KMeans clustering for unsupervised subgroup discovery
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=clusters, palette="Set2")
plt.title("KMeans Clustering of Students")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()






